In [25]:

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from imblearn.under_sampling import RandomUnderSampler 
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
import numpy as np
from sklearn.metrics import accuracy_score
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from collections import defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score  
import scipy
rseed = 6740
np.random.seed(rseed)

In [26]:
class loadData:
    def __init__(self,randomstate=6740):
        self.test_data_fashion = pd.read_csv('data/fashion-mnist_test.csv')                              
        self.train_data_fashion = pd.read_csv('data/fashion-mnist_train.csv')                            
        self.data_digits = scipy.io.loadmat('data/mnist_10digits.mat')                                   
                                                                                                        
        self.train_data_x_fashion = self.train_data_fashion.iloc[:, 1:]                                  
        self.train_data_y_fashion = self.train_data_fashion.iloc[:, 0]                                   
        self.test_data_x_fashion  = self.test_data_fashion.iloc[:, 1:]                                   
        self.test_data_y_fashion  = self.test_data_fashion.iloc[:, 0]                                    

        self.train_data_x_digits = self.data_digits['xtrain']                                            
        self.train_data_y_digits = self.data_digits['ytrain']                                          
        self.test_data_x_digits  = self.data_digits['xtest']                                             
        self.test_data_y_digits  = self.data_digits['ytest']                                             
                                                                                                        
        scaler_fashion = MinMaxScaler()                                                                  
        scaler_digits  = MinMaxScaler()                                                                  
                                                                                                        
        self.fashion_train_data_x_scaled = scaler_fashion.fit_transform(self.train_data_x_fashion)       
        self.fashion_test_data_x_scaled  = scaler_fashion.transform(self.test_data_x_fashion)            
                                                                                                        
        self.digits_train_data_x_scaled = scaler_digits.fit_transform(self.train_data_x_digits)          
        self.digits_test_data_x_scaled  = scaler_digits.transform(self.test_data_x_digits)

   


    

In [27]:
data_sets = loadData()

In [28]:
def logisitc_regression(data_x,data_y):
    log_reg_model = LogisticRegression(random_state=6740,max_iter=2000)
    log_reg_model.fit(data_x,data_y)
    return log_reg_model

def knn(data_x,data_y,k):
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(data_x,data_y)
    return knn_model

def mlp(data_x,data_y):
    mlp_model = MLPClassifier(hidden_layer_sizes=(20,10),random_state=6740)
    mlp_model.fit(data_x, data_y)
    return mlp_model

def svm(data_x,data_y,kernel=False):
    rand_downsample = np.random.choice(data_x.shape[0], 5000, replace=False) 
    data_x_downsampled = data_x[rand_downsample]
    data_y_downsampled = data_y[rand_downsample]
    if kernel ==False:
        svm_model = SVC(kernel='linear',random_state=6740)
    else:
        svm_model = SVC(kernel='rbf',random_state=6740)
    
    svm_model.fit(data_x_downsampled,data_y_downsampled)

    return svm_model

    
    




In [29]:
fashion_model_dict = {}

In [30]:
fashion_model_dict = {}
digit_model_dict = {}
fashion_model_dict['logistic_regression']=logisitc_regression(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)
fashion_model_dict['mlp']=mlp(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)
fashion_model_dict['knn']=knn(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion,4)
fashion_model_dict['svm']=svm(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)
fashion_model_dict['rbf']=svm(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion,kernel=True)

digit_model_dict['logistic_regression']=logisitc_regression(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0])
digit_model_dict['mlp']=mlp(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0])
digit_model_dict['knn']=knn(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0],4)
digit_model_dict['svm']=svm(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0])
digit_model_dict['rbf']=svm(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0],kernel=True)

/Users/willbrennan/Desktop/Coding/school_repo/school_code/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/willbrennan/Desktop/Coding/school_repo/school_code/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [31]:
def compute_metrics(y_true,y_pred):

    precision=precision_score(y_true,y_pred,average='weighted')
    recall=recall_score(y_true,y_pred,average='weighted')
    f1=f1_score(y_true,y_pred,average='weighted')
    return [precision,recall,f1]

In [32]:
fashion_metric_list = []
digit_metric_list = []
for fashion,digits in zip(fashion_model_dict.items(),digit_model_dict.items()):

    fashion_model_predictions = fashion[1].predict(data_sets.fashion_test_data_x_scaled)
    digit_model_predictions = digits[1].predict(data_sets.digits_test_data_x_scaled)

    fashion_metric_list.append(compute_metrics(data_sets.test_data_y_fashion,fashion_model_predictions)+[fashion[0]])
    digit_metric_list.append(compute_metrics(data_sets.test_data_y_digits[0],digit_model_predictions)+[digits[0]])

In [33]:
pd.DataFrame(fashion_metric_list,columns=['precision','recall','f1','model']).set_index('model')

,precision,recall,f1
model,,,
logistic_regression,0.852297,0.8538,0.852854
mlp,0.866694,0.8681,0.867032
knn,0.864130,0.8618,0.860244
svm,0.814821,0.8163,0.815152
rbf,0.851727,0.8525,0.850611


In [34]:
pd.DataFrame(digit_metric_list,columns=['precision','recall','f1','model']).set_index('model')

,precision,recall,f1
model,,,
logistic_regression,0.925809,0.9260,0.925833
mlp,0.948409,0.9484,0.948356
knn,0.968520,0.9682,0.968129
svm,0.911205,0.9113,0.910932
rbf,0.953092,0.9531,0.953030


In [36]:
# param_grid = {'n_neighbors': range(1, 21)}

# knn_model = KNeighborsClassifier()


# grid_search = GridSearchCV(knn_model, param_grid, cv=5)
# grid_search.fit(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)

# grid_search.best_params_['n_neighbors']

